# 02 — Chargement de la base SIRENE

Charge depuis les parquets INSEE :
- **Etab actifs** (vue `etab_active`) : pour la siretisation
- **UL + adresse siège** (vue `etab_siege`) : pour la sirenisation

Sortie : 2 fichiers parquet dans `data/interim/`.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from src.connexion   import get_duckdb_connection
from src.display     import afficher_tableau
from config.settings import (
    SIRENE_ETAB_RAW, SIRENE_UL_RAW, INTERIM_DIR,
)

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

## 1. Établissements actifs (siretisation)

Vue `etab_active` : tous les établissements actifs avec dénomination UL et enseignes établissement.

In [3]:
duckdb_con = get_duckdb_connection()

duckdb_con.execute(f"""
    COPY (
        SELECT siren, siret, nic,
               denominationUniteLegale, sigleUniteLegale,
               categorieJuridiqueUniteLegale, activitePrincipaleUniteLegale,
               dateCreationUniteLegale,
               etablissementSiege,
               dateCreationEtablissement,
               enseigne1Etablissement, enseigne2Etablissement,
               enseigne3Etablissement, denominationUsuelleEtablissement,
               numeroVoieEtablissement, typeVoieEtablissement,
               libelleVoieEtablissement, codeCommuneEtablissement
        FROM etab_active
    ) TO '{SIRENE_ETAB_RAW}' (FORMAT PARQUET)
""")
print(f'Établissements SIRENE écrits → {SIRENE_ETAB_RAW}')

# Aperçu léger : on relit juste 5 lignes du parquet, pas tout en mémoire
afficher_tableau(pd.read_parquet(SIRENE_ETAB_RAW).head(), 'Aperçu Etab SIRENE')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Établissements SIRENE écrits → /home/jovyan/work/projet_finess_sirene/data/interim/sirene_etab.parquet


siren,siret,nic,denominationUniteLegale,sigleUniteLegale,categorieJuridiqueUniteLegale,activitePrincipaleUniteLegale,dateCreationUniteLegale,etablissementSiege,dateCreationEtablissement,enseigne1Etablissement,enseigne2Etablissement,enseigne3Etablissement,denominationUsuelleEtablissement,numeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,codeCommuneEtablissement
000325175,00032517500065,00065,None,None,1000,32.12Z,2000-09-26,True,2018-02-07,None,None,None,None,51,RUE,MARX DORMOY,13204
005420021,00542002100056,00056,ETABLISSEMENTS LUCIEN BIQUEZ,None,5710,46.69B,1954-01-01,True,2009-12-23,None,None,None,None,21,BOULEVARD,DES PRES,80001
005420120,00542012000015,00015,SOCIETE DES SUCRERIES DU MARQUENTERRE,None,5599,70.10Z,1954-01-01,False,1989-01-27,None,None,None,None,None,RUE,DE LA FONTAINE,80688
005420120,00542012000023,00023,SOCIETE DES SUCRERIES DU MARQUENTERRE,None,5599,70.10Z,1954-01-01,False,1900-01-01,None,None,None,None,12,ROUTE,DE MONTREUIL,62688
005420120,00542012000056,00056,SOCIETE DES SUCRERIES DU MARQUENTERRE,None,5599,70.10Z,1954-01-01,True,2026-01-13,None,None,None,None,32,CHEMIN,DES GARENNES,80713


## 2. UL avec adresse siège (sirenisation)

Vue `etab_siege` : une ligne par UL active, avec l'adresse de son siège.

In [5]:
duckdb_con.execute(f"""
    COPY (
        SELECT siren, denominationUniteLegale, sigleUniteLegale,
               categorieJuridiqueUniteLegale, activitePrincipaleUniteLegale,
               dateCreationUniteLegale,
               nicSiegeUniteLegale,
               numeroVoieEtablissement, typeVoieEtablissement,
               libelleVoieEtablissement, codeCommuneEtablissement
        FROM etab_siege
    ) TO '{SIRENE_UL_RAW}' (FORMAT PARQUET)
""")
duckdb_con.close()
print(f'UL SIRENE écrites → {SIRENE_UL_RAW}')

afficher_tableau(pd.read_parquet(SIRENE_UL_RAW).head(), 'Aperçu UL SIRENE')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

UL SIRENE écrites → /home/jovyan/work/projet_finess_sirene/data/interim/sirene_ul.parquet


siren,denominationUniteLegale,sigleUniteLegale,categorieJuridiqueUniteLegale,activitePrincipaleUniteLegale,dateCreationUniteLegale,nicSiegeUniteLegale,numeroVoieEtablissement,typeVoieEtablissement,libelleVoieEtablissement,codeCommuneEtablissement
101753002,[ND],[ND],1000,01.42Z,2026-02-26,00019,[ND],[ND],[ND],05011
101753010,JSI,None,6540,68.20B,2026-02-21,00012,15,RUE,DE LIMOURS,78128
101753028,None,None,1000,62.01Z,2026-04-01,00014,1,RUE,DES JONQUILLES,34176
101753036,INDIVISION VERDAGUER RAMBEAU,None,2110,55.20Z,2026-03-01,00017,42,RUE,DES FRERES COLIN,14118
101753044,[ND],[ND],1000,70.22Z,2026-02-26,00011,[ND],[ND],[ND],92035
